<a href="https://colab.research.google.com/github/Manish927/EDA-Data-Science/blob/main/Ensemble_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier

In [10]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
Y = pd.Series(data.target)

In [6]:
X.shape

(569, 30)

In [7]:
X.columns

Index(['mean radius', 'mean texture', 'mean perimeter', 'mean area',
       'mean smoothness', 'mean compactness', 'mean concavity',
       'mean concave points', 'mean symmetry', 'mean fractal dimension',
       'radius error', 'texture error', 'perimeter error', 'area error',
       'smoothness error', 'compactness error', 'concavity error',
       'concave points error', 'symmetry error', 'fractal dimension error',
       'worst radius', 'worst texture', 'worst perimeter', 'worst area',
       'worst smoothness', 'worst compactness', 'worst concavity',
       'worst concave points', 'worst symmetry', 'worst fractal dimension'],
      dtype='object')

In [ ]:
X.dtypes

In [11]:
Y.dtypes

dtype('int64')

In [ ]:
X.isnull().sum()

In [37]:
# train test split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=41)

In [38]:
print(X_train.shape, X_test.shape, Y_train.shape, Y_test.shape)

(455, 30) (114, 30) (455,) (114,)


In [39]:
# start with a baseline model, i.e train Baseline Decision Tree
modelDt = DecisionTreeClassifier()  # initializing
modelDt.fit(X_train, Y_train) # training
Y_predDt = modelDt.predict(X_test) # predicting


In [40]:
# classisifcation report
print("Decision Tree Accuracy:", accuracy_score(Y_test, Y_predDt))
print(classification_report(Y_test, Y_predDt))

Decision Tree Accuracy: 0.956140350877193
              precision    recall  f1-score   support

           0       0.91      0.97      0.94        40
           1       0.99      0.95      0.97        74

    accuracy                           0.96       114
   macro avg       0.95      0.96      0.95       114
weighted avg       0.96      0.96      0.96       114



In [ ]:
#print confusion_matrix
print("Confusion Matrix:", confusion_matrix(Y_test, Y_predDt))


Train Random Forest, because decision tree never goes back, it is from top to bottom, if a good root is selected then the tree may not overfit, hence we need to take a chance to explore other tree. So Random Forest is one of the bagging algorithm

Bagging and Boosting, both are ensemble approach

In [45]:
rf = RandomForestClassifier()
rf.fit(X_train, Y_train)

RandomForestClassifier()

In [46]:
y_pred_rf = rf.predict(X_test)
print("Random Forest Accuracy:", accuracy_score(Y_test, y_pred_rf))

Random Forest Accuracy: 0.9824561403508771


In [51]:
# give rf parameters using GridSearchCV to push hyper parameter combination and giving me a best result
param_grid = {"n_estimators": [100, 200], # no of trees
              "max_depth": [5, 10, None],
              "min_samples_split": [2, 5],
              "criterion": ["gini", "entropy"]
}

In [52]:
est = RandomForestClassifier(random_state = 41)
rf_grid = GridSearchCV(estimator=est, param_grid=param_grid, cv=5)
rf_grid.fit(X_train, Y_train)
print("Best RF Params:", rf_grid.best_params_)

Best RF Params: {'criterion': 'entropy', 'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 200}


In [54]:
best_rf = rf_grid.best_estimator_
y_pred_rf = best_rf.predict(X_test)
print("Random Forest Accuracy:", accuracy_score(Y_test, y_pred_rf))
print(classification_report(Y_test, y_pred_rf))

Random Forest Accuracy: 0.9824561403508771
              precision    recall  f1-score   support

           0       0.95      1.00      0.98        40
           1       1.00      0.97      0.99        74

    accuracy                           0.98       114
   macro avg       0.98      0.99      0.98       114
weighted avg       0.98      0.98      0.98       114



Next we will use the AdaBoost Algorithm, currently this is less used, gradient boost, XGBoost are extensively used.
AdaBoost max-depth is only 1


In [55]:
# Train AdaBoost
ada = AdaBoostClassifier(random_state=41)
ada.fit(X_train, Y_train)

AdaBoostClassifier(random_state=41)

In [95]:
#y_pred_ada = ada.best_estimator_
y_pred_ada = ada.predict(X_test)
print("AdaBoost Accuracy:", accuracy_score(Y_test, y_pred_ada))


AdaBoost Accuracy: 0.9912280701754386


In [58]:
ada_params = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1.0]
}

In [59]:
est = AdaBoostClassifier(random_state=41)
ada_grid = GridSearchCV(estimator=est, param_grid=ada_params, cv=5)
ada_grid.fit(X_train, Y_train)
print("Best AdaBoost Params:", ada_grid.best_params_)

Best AdaBoost Params: {'learning_rate': 1.0, 'n_estimators': 200}


In [60]:
best_ada = ada_grid.best_estimator_
y_pred_ada = best_ada.predict(X_test)
print("AdaBoost Accuracy:", accuracy_score(Y_test, y_pred_ada))
print(classification_report(Y_test, y_pred_ada))

AdaBoost Accuracy: 0.9824561403508771
              precision    recall  f1-score   support

           0       0.95      1.00      0.98        40
           1       1.00      0.97      0.99        74

    accuracy                           0.98       114
   macro avg       0.98      0.99      0.98       114
weighted avg       0.98      0.98      0.98       114



Now we will try with Gradient Boosting

In [61]:
gb = GradientBoostingClassifier(random_state=41)
gb.fit(X_train, Y_train)


GradientBoostingClassifier(random_state=41)

In [96]:
best_gb = ada_grid.best_estimator_
y_pred_gb = gb.predict(X_test)
print("Gradient Boosting Accuracy:", accuracy_score(Y_test, y_pred_gb))

Gradient Boosting Accuracy: 0.9824561403508771


In [66]:
# deing the gb_params
gb_params = {
    "n_estimators": [100, 100,],
    "learning_rate": [0.01, 0.1],
    "max_depth": [3, 5],
    "min_samples_split": [2, 5],
}

In [67]:
est = GradientBoostingClassifier(random_state=41)
gb_grid = GridSearchCV(estimator=est, param_grid=gb_params)
gb_grid.fit(X_train, Y_train)
print("Best Gradient Boosting Params:", gb_grid.best_params_)
print("Best Gradient Boosting Score:", gb_grid.best_score_)
print("Gradient Boosting Score:", gb_grid.score(X_test, Y_test))
print("Gradient Boosting Accuracy:", accuracy_score(Y_test, gb_grid.predict(X_test)))


Best Gradient Boosting Params: {'learning_rate': 0.1, 'max_depth': 3, 'min_samples_split': 5, 'n_estimators': 100}
Best Gradient Boosting Score: 0.956043956043956
Gradient Boosting Score: 0.9649122807017544
Gradient Boosting Accuracy: 0.9649122807017544


In [97]:
best_gb = gb_grid.best_estimator_
y_pred_gb = best_gb.predict(X_test)
print("Gradient Boosting Accuracy:", accuracy_score(Y_test, y_pred_gb))
print(classification_report(Y_test, y_pred_gb))

Gradient Boosting Accuracy: 0.9649122807017544
              precision    recall  f1-score   support

           0       0.91      1.00      0.95        40
           1       1.00      0.95      0.97        74

    accuracy                           0.96       114
   macro avg       0.95      0.97      0.96       114
weighted avg       0.97      0.96      0.97       114



### Train XGBoost

In [68]:
xgb = XGBClassifier(random_state=41)
xgb.fit(X_train, Y_train)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [98]:
y_pred_xgb = xgb.predict(X_test)
print("XGBoost Accuracy:", accuracy_score(Y_test, y_pred_xgb))

XGBoost Accuracy: 0.9912280701754386


In [80]:
xgb_params = {
    "n_estimators": [100, 200],
    "learning_rate": [0.01, 0.1],
    "max_depth": [3, 5, 7],
}

In [81]:
est = XGBClassifier()
xgb_grid = GridSearchCV(estimator=est, param_grid=xgb_params)
xgb_grid.fit(X_train, Y_train)
print("Best XGBoost Params:", xgb_grid.best_params_)
print("Best XGBoost Score:", xgb_grid.best_score_)
print("XGBoost Score:", xgb_grid.score(X_test, Y_test))
print("XGBoost Accuracy:", accuracy_score(Y_test, xgb_grid.predict(X_test)))


Best XGBoost Params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200}
Best XGBoost Score: 0.9714285714285715
XGBoost Score: 0.9912280701754386
XGBoost Accuracy: 0.9912280701754386


In [99]:
best_xgb = xgb_grid.best_estimator_
y_pred_xgb = best_xgb.predict(X_test)
print("XGBoost Accuracy:", accuracy_score(Y_test, y_pred_xgb))
print(classification_report(Y_test, y_pred_xgb))

XGBoost Accuracy: 0.9912280701754386
              precision    recall  f1-score   support

           0       0.98      1.00      0.99        40
           1       1.00      0.99      0.99        74

    accuracy                           0.99       114
   macro avg       0.99      0.99      0.99       114
weighted avg       0.99      0.99      0.99       114



In [84]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score


Summary of All Algorithm

In [101]:
# print the accuracy of all the model
y_pred_dt = modelDt.predict(X_test)
y_pred_rf = best_rf.predict(X_test)
y_pred_ada = best_ada.predict(X_test)
y_pred_gb = best_gb.predict(X_test)
y_pred_xgb = best_xgb.predict(X_test)

In [104]:
results = pd.DataFrame({
    "Model": ["Decision Tree", "Random Forest", "AdaBoost", "Gradient Boosting", "XGBoost"],

    "Accuracy": [
        accuracy_score(Y_test, y_pred_dt),
        accuracy_score(Y_test, y_pred_rf),
        accuracy_score(Y_test, y_pred_ada),
        accuracy_score(Y_test, y_pred_gb),
        accuracy_score(Y_test, y_pred_xgb)
    ],

    "Precision": [
        precision_score(Y_test, y_pred_dt),
        precision_score(Y_test, y_pred_rf),
        precision_score(Y_test, y_pred_ada),
        precision_score(Y_test, y_pred_gb),
        precision_score(Y_test, y_pred_xgb)
    ],

    "Recall": [
        recall_score(Y_test, y_pred_dt),
        recall_score(Y_test, y_pred_rf),
        recall_score(Y_test, y_pred_ada),
        recall_score(Y_test, y_pred_gb),
        recall_score(Y_test, y_pred_xgb)
    ],

    "F1 Score": [
        f1_score(Y_test, y_pred_dt),
        f1_score(Y_test, y_pred_rf),
        f1_score(Y_test, y_pred_ada),
        f1_score(Y_test, y_pred_gb),
        f1_score(Y_test, y_pred_xgb)
    ],

    "ROC AUC": [
        roc_auc_score(Y_test, y_pred_dt),
        roc_auc_score(Y_test, y_pred_rf),
        roc_auc_score(Y_test, y_pred_ada),
        roc_auc_score(Y_test, y_pred_gb),
        roc_auc_score(Y_test, y_pred_xgb)
    ]
})

results.round(2)

,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,Decision Tree,0.96,0.99,0.95,0.97,0.96
1,Random Forest,0.98,1.00,0.97,0.99,0.99
2,AdaBoost,0.98,1.00,0.97,0.99,0.99
3,Gradient Boosting,0.96,1.00,0.95,0.97,0.97
4,XGBoost,0.99,1.00,0.99,0.99,0.99
